# End-to-End Sales Forecasting & Demand Intelligence System
**Superstore Sales Dataset (2015-2018)** | Time Series Analysis, Forecasting, Anomaly Detection, Segmentation

In [31]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100
plt.rcParams['figure.figsize'] = (11, 5)
import os
os.makedirs('charts', exist_ok=True)

In [32]:
import sys
!{sys.executable} -m pip install xgboost statsmodels scikit-learn plotly streamlit

## Task 1 — Data Loading, Exploration & Feature Engineering

In [33]:
df = pd.read_csv('train.csv')
df['Order Date'] = pd.to_datetime(df['Order Date'], dayfirst=True)
df['Ship Date'] = pd.to_datetime(df['Ship Date'], dayfirst=True)
df['Year'] = df['Order Date'].dt.year
df['Month'] = df['Order Date'].dt.month
df['Week'] = df['Order Date'].dt.isocalendar().week
df['DayOfWeek'] = df['Order Date'].dt.day_name()
df['Quarter'] = df['Order Date'].dt.quarter

def season(m):
    if m in [12, 1, 2]: return 'Winter'
    if m in [3, 4, 5]: return 'Spring'
    if m in [6, 7, 8]: return 'Summer'
    return 'Autumn'

df['Season'] = df['Month'].apply(season)
df['ShipDelay'] = (df['Ship Date'] - df['Order Date']).dt.days

weekly = df.set_index('Order Date').resample('W-MON')['Sales'].sum().reset_index()
monthly = df.set_index('Order Date').resample('MS')['Sales'].sum().reset_index()

In [34]:
cat_rev = df.groupby('Category')['Sales'].sum().sort_values(ascending=False)
plt.figure(figsize=(8,5))
sns.barplot(x=cat_rev.index, y=cat_rev.values, hue=cat_rev.index, palette='Blues_d', legend=False)
plt.title('Total Revenue by Category', fontsize=13, fontweight='bold')
plt.ylabel('Sales ($)')
plt.tight_layout()
plt.savefig('charts/01_category_revenue.png', dpi=150)
plt.close()

In [35]:
region_yearly = df.groupby(['Region', 'Year'])['Sales'].sum().reset_index()
consistency = {}
for r in region_yearly['Region'].unique():
    sub = region_yearly[region_yearly['Region'] == r].sort_values('Year')
    yoy = sub['Sales'].pct_change().dropna()
    consistency[r] = yoy.std()
consistency_df = pd.Series(consistency).sort_values()

In [36]:
ship_overall = df['ShipDelay'].mean()
month_totals = df.groupby('Month')['Sales'].sum().sort_values(ascending=False)
plt.figure(figsize=(9,5))
sns.barplot(x=month_totals.index, y=month_totals.values, hue=month_totals.index, palette='Oranges_d', legend=False)
plt.title('Total Sales by Calendar Month (All Years)', fontsize=13, fontweight='bold')
plt.xlabel('Month')
plt.ylabel('Sales ($)')
plt.tight_layout()
plt.savefig('charts/02_monthly_seasonality.png', dpi=150)
plt.close()

## Task 2 — Time Series Decomposition & Stationarity Testing

In [37]:
monthly_ts = monthly.set_index('Order Date').asfreq('MS')
plt.figure(figsize=(12,5))
plt.plot(monthly_ts.index, monthly_ts['Sales'], marker='o', color='#2563eb', linewidth=2)
plt.title('Monthly Sales Trend (2015-2018)', fontsize=13, fontweight='bold')
plt.xlabel('Month')
plt.ylabel('Sales ($)')
plt.tight_layout()
plt.savefig('charts/03_monthly_trend.png', dpi=150)
plt.close()

In [38]:
from statsmodels.tsa.seasonal import seasonal_decompose
decomp = seasonal_decompose(monthly_ts['Sales'], model='additive', period=12)
fig, axes = plt.subplots(4, 1, figsize=(12,10), sharex=True)
decomp.observed.plot(ax=axes[0], color='#2563eb')
axes[0].set_ylabel('Observed', fontweight='bold')
decomp.trend.plot(ax=axes[1], color='#16a34a')
axes[1].set_ylabel('Trend', fontweight='bold')
decomp.seasonal.plot(ax=axes[2], color='#d97706')
axes[2].set_ylabel('Seasonal', fontweight='bold')
decomp.resid.plot(ax=axes[3], color='#dc2626', marker='o', linestyle='None')
axes[3].set_ylabel('Residual', fontweight='bold')
axes[0].set_title('Additive Decomposition — Monthly Sales', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('charts/04_decomposition.png', dpi=150)
plt.close()

In [39]:
from statsmodels.tsa.stattools import adfuller
adf_raw = adfuller(monthly_ts['Sales'].dropna())
diff_series = monthly_ts['Sales'].diff().dropna()
adf_diff = adfuller(diff_series)

## Task 3 — Sales Forecasting using 3 Different Models

In [40]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

def mape(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    return np.mean(np.abs((y_true - y_pred) / y_true)) * 100

series = monthly_ts['Sales']
train_s, test_s = series.iloc[:-3], series.iloc[-3:]

In [41]:
from statsmodels.tsa.statespace.sarimax import SARIMAX
sarima_model = SARIMAX(train_s, order=(1,1,1), seasonal_order=(1,1,1,12),
                        enforce_stationarity=False, enforce_invertibility=False)
sarima_fit = sarima_model.fit(disp=False)
sarima_fc = sarima_fit.get_forecast(steps=3)
sarima_pred = sarima_fc.predicted_mean
sarima_ci = sarima_fc.conf_int()
sarima_mae = mean_absolute_error(test_s, sarima_pred)
sarima_rmse = np.sqrt(mean_squared_error(test_s, sarima_pred))
sarima_mape = mape(test_s, sarima_pred)

In [42]:
plt.figure(figsize=(11,5))
plt.plot(train_s.index, train_s, label='Train Actual', color='#2563eb', linewidth=2)
plt.plot(test_s.index, test_s, label='Test Actual', color='#16a34a', marker='o', linewidth=2)
plt.plot(sarima_pred.index, sarima_pred, label='SARIMA Forecast', color='#dc2626', marker='o', linestyle='--', linewidth=2)
plt.fill_between(sarima_ci.index, sarima_ci.iloc[:,0], sarima_ci.iloc[:,1], color='#dc2626', alpha=0.15)
plt.title('SARIMA: Actual vs Forecasted Monthly Sales', fontsize=13, fontweight='bold')
plt.legend()
plt.tight_layout()
plt.savefig('charts/05_sarima_forecast.png', dpi=150)
plt.close()

In [43]:
from prophet import Prophet
import logging
logging.getLogger('prophet').setLevel(logging.ERROR)
logging.getLogger('cmdstanpy').setLevel(logging.ERROR)

prophet_df = monthly_ts.reset_index()[['Order Date', 'Sales']].rename(columns={'Order Date': 'ds', 'Sales': 'y'})
prophet_train, prophet_test = prophet_df.iloc[:-3], prophet_df.iloc[-3:]
pm = Prophet(yearly_seasonality=True, weekly_seasonality=False, daily_seasonality=False, seasonality_mode='multiplicative')
pm.fit(prophet_train)
future = pm.make_future_dataframe(periods=3, freq='MS')
forecast = pm.predict(future)
test_fc = forecast.iloc[-3:]
prophet_mae = mean_absolute_error(prophet_test['y'], test_fc['yhat'])
prophet_rmse = np.sqrt(mean_squared_error(prophet_test['y'], test_fc['yhat']))
prophet_mape = mape(prophet_test['y'], test_fc['yhat'])

In [44]:
fig1 = pm.plot(forecast)
plt.title('Prophet Forecast — Monthly Sales', fontsize=13, fontweight='bold')
plt.tight_layout()
fig1.savefig('charts/06_prophet_forecast.png', dpi=150)
plt.close()
fig2 = pm.plot_components(forecast)
fig2.savefig('charts/07_prophet_components.png', dpi=150)
plt.close()

In [45]:
ml_df = monthly_ts.reset_index()[['Order Date', 'Sales']]
ml_df['Month'] = ml_df['Order Date'].dt.month
ml_df['Quarter'] = ml_df['Order Date'].dt.quarter
def season_num(m):
    if m in [12,1,2]: return 0
    if m in [3,4,5]: return 1
    if m in [6,7,8]: return 2
    return 3
ml_df['Season'] = ml_df['Month'].apply(season_num)
ml_df['Lag1'] = ml_df['Sales'].shift(1)
ml_df['Lag2'] = ml_df['Sales'].shift(2)
ml_df['Lag3'] = ml_df['Sales'].shift(3)
ml_df['RollingMean3'] = ml_df['Sales'].shift(1).rolling(3).mean()
ml_df = ml_df.dropna().reset_index(drop=True)
features = ['Lag1', 'Lag2', 'Lag3', 'RollingMean3', 'Month', 'Quarter', 'Season']
ml_train, ml_test = ml_df.iloc[:-3], ml_df.iloc[-3:]

In [46]:
from xgboost import XGBRegressor
xgb_model = XGBRegressor(n_estimators=200, max_depth=3, learning_rate=0.05, random_state=42)
xgb_model.fit(ml_train[features], ml_train['Sales'])
lag_vals = ml_df['Sales'].iloc[len(ml_train)-3:len(ml_train)].tolist()
xgb_preds = []
for i in range(3):
    row = ml_test.iloc[i]
    lag1, lag2, lag3 = lag_vals[-1], lag_vals[-2], lag_vals[-3]
    roll = np.mean(lag_vals[-3:])
    X_row = pd.DataFrame([[lag1, lag2, lag3, roll, row['Month'], row['Quarter'], row['Season']]], columns=features)
    pred = xgb_model.predict(X_row)[0]
    xgb_preds.append(pred)
    lag_vals.append(pred)
xgb_mae = mean_absolute_error(ml_test['Sales'], xgb_preds)
xgb_rmse = np.sqrt(mean_squared_error(ml_test['Sales'], xgb_preds))
xgb_mape = mape(ml_test['Sales'], xgb_preds)

In [47]:
plt.figure(figsize=(11,5))
plt.plot(ml_train['Order Date'], ml_train['Sales'], label='Train Actual', color='#2563eb', linewidth=2)
plt.plot(ml_test['Order Date'], ml_test['Sales'], label='Test Actual', color='#16a34a', marker='o', linewidth=2)
plt.plot(ml_test['Order Date'], xgb_preds, label='XGBoost Forecast', color='#dc2626', marker='o', linestyle='--', linewidth=2)
plt.title('XGBoost: Actual vs Predicted Monthly Sales', fontsize=13, fontweight='bold')
plt.legend()
plt.tight_layout()
plt.savefig('charts/08_xgboost_forecast.png', dpi=150)
plt.close()

In [48]:
comparison = pd.DataFrame({
    'Model': ['SARIMA', 'Prophet', 'XGBoost'],
    'MAE': [sarima_mae, prophet_mae, xgb_mae],
    'RMSE': [sarima_rmse, prophet_rmse, xgb_rmse],
    'MAPE (%)': [sarima_mape, prophet_mape, xgb_mape]
}).set_index('Model').round(2)

## Task 4 — Product Category & Region Level Forecasting

In [49]:
def build_monthly(d):
    m = d.set_index('Order Date').resample('MS')['Sales'].sum().reset_index()
    m.columns = ['Order Date', 'Sales']
    return m

def make_features(m):
    d = m.copy()
    d['Month'] = d['Order Date'].dt.month
    d['Quarter'] = d['Order Date'].dt.quarter
    d['Season'] = d['Month'].apply(season_num)
    d['Lag1'] = d['Sales'].shift(1)
    d['Lag2'] = d['Sales'].shift(2)
    d['Lag3'] = d['Sales'].shift(3)
    d['RollingMean3'] = d['Sales'].shift(1).rolling(3).mean()
    return d.dropna().reset_index(drop=True)

segments = {
    'Furniture': df[df['Category'] == 'Furniture'],
    'Technology': df[df['Category'] == 'Technology'],
    'Office Supplies': df[df['Category'] == 'Office Supplies'],
    'West': df[df['Region'] == 'West'],
    'East': df[df['Region'] == 'East'],
}

segment_results = {}
plt.figure(figsize=(12,6))
colors = ['#2563eb', '#16a34a', '#d97706', '#dc2626', '#7c3aed']

for (name, d), c in zip(segments.items(), colors):
    m = build_monthly(d)
    feat_df = make_features(m)
    s_train, s_test = feat_df.iloc[:-3], feat_df.iloc[-3:]
    model = XGBRegressor(n_estimators=150, max_depth=3, learning_rate=0.05, random_state=42)
    model.fit(s_train[features], s_train['Sales'])
    lags = feat_df['Sales'].iloc[len(s_train)-3:len(s_train)].tolist()
    preds = []
    for i in range(3):
        row = s_test.iloc[i]
        lag1, lag2, lag3 = lags[-1], lags[-2], lags[-3]
        roll = np.mean(lags[-3:])
        X_row = pd.DataFrame([[lag1, lag2, lag3, roll, row['Month'], row['Quarter'], row['Season']]], columns=features)
        p = model.predict(X_row)[0]
        preds.append(p)
        lags.append(p)
    growth = (preds[-1] - m['Sales'].iloc[-1]) / m['Sales'].iloc[-1] * 100
    segment_results[name] = {'forecast': preds, 'growth_pct': growth}
    plt.plot(m['Order Date'], m['Sales'], color=c, alpha=0.35)
    plt.plot(s_test['Order Date'], preds, color=c, marker='o', linestyle='--', label=name, linewidth=2)

plt.title('3-Month Forecast by Category & Region', fontsize=13, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Sales ($)')
plt.legend(loc='upper left', fontsize=9)
plt.tight_layout()
plt.savefig('charts/09_segment_forecasts.png', dpi=150)
plt.close()

## Task 5 — Anomaly Detection in Sales Data

In [50]:
from sklearn.ensemble import IsolationForest
weekly_an = weekly.copy()
iso = IsolationForest(contamination=0.06, random_state=42)
weekly_an['iso_flag'] = iso.fit_predict(weekly_an[['Sales']]) == -1
roll_mean = weekly_an['Sales'].rolling(8, center=True, min_periods=1).mean()
roll_std = weekly_an['Sales'].rolling(8, center=True, min_periods=1).std()
weekly_an['zscore'] = (weekly_an['Sales'] - roll_mean) / roll_std
weekly_an['z_flag'] = weekly_an['zscore'].abs() > 2

In [51]:
fig, axes = plt.subplots(2, 1, figsize=(12,9), sharex=True)
axes[0].plot(weekly_an['Order Date'], weekly_an['Sales'], color='#2563eb', label='Weekly Sales', linewidth=1.5)
axes[0].scatter(weekly_an[weekly_an['iso_flag']]['Order Date'], weekly_an[weekly_an['iso_flag']]['Sales'],
                color='#dc2626', s=60, zorder=5, marker='X', label='Isolation Forest Anomaly')
axes[0].set_title('Isolation Forest: Weekly Sales Anomalies', fontsize=12, fontweight='bold')
axes[0].legend()
axes[1].plot(weekly_an['Order Date'], weekly_an['Sales'], color='#2563eb', label='Weekly Sales', linewidth=1.5)
axes[1].scatter(weekly_an[weekly_an['z_flag']]['Order Date'], weekly_an[weekly_an['z_flag']]['Sales'],
                color='#d97706', s=60, zorder=5, marker='D', label='Z-Score Anomaly (>2σ)')
axes[1].set_title('Z-Score Method: Weekly Sales Anomalies', fontsize=12, fontweight='bold')
axes[1].legend()
plt.tight_layout()
plt.savefig('charts/10_anomaly_detection.png', dpi=150)
plt.close()

## Task 6 — Product Demand Segmentation using Clustering

In [52]:
monthly_sc = df.groupby(['Sub-Category', pd.Grouper(key='Order Date', freq='MS')])['Sales'].sum().reset_index()
feats = []
for sc, g in monthly_sc.groupby('Sub-Category'):
    g = g.sort_values('Order Date')
    total_sales = g['Sales'].sum()
    volatility = g['Sales'].std()
    yearly = g.set_index('Order Date').resample('YS')['Sales'].sum()
    yoy = yearly.pct_change().dropna()
    growth_rate = yoy.mean() if len(yoy) > 0 else 0
    order_count = df[df['Sub-Category'] == sc].shape[0]
    avg_order_value = total_sales / order_count
    feats.append({'Sub-Category': sc, 'TotalSales': total_sales, 'GrowthRate': growth_rate,
                   'Volatility': volatility, 'AvgOrderValue': avg_order_value})
feat_df = pd.DataFrame(feats).set_index('Sub-Category')

In [53]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

X = feat_df.fillna(0)
X_scaled = StandardScaler().fit_transform(X)
inertias = []
for k in range(1, 8):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertias.append(km.inertia_)

plt.figure(figsize=(8,5))
plt.plot(range(1, 8), inertias, marker='o', color='#2563eb', linewidth=2, markersize=8)
plt.title('Elbow Method for Optimal K', fontsize=13, fontweight='bold')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Inertia')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('charts/11_elbow_method.png', dpi=150)
plt.close()

In [54]:
k_opt = 4
km = KMeans(n_clusters=k_opt, random_state=42, n_init=10)
feat_df['Cluster'] = km.fit_predict(X_scaled)
pca = PCA(n_components=2)
coords = pca.fit_transform(X_scaled)
feat_df['PC1'] = coords[:, 0]
feat_df['PC2'] = coords[:, 1]

cluster_labels = {
    0: 'High-Value, Volatile',
    1: 'Low Volume, Stable',
    2: 'High Volume, Stable',
    3: 'Growing Demand',
}
feat_df['ClusterLabel'] = feat_df['Cluster'].map(cluster_labels)

plt.figure(figsize=(10,7))
colors = ['#2563eb', '#16a34a', '#d97706', '#dc2626']
for c in sorted(feat_df['Cluster'].unique()):
    sub = feat_df[feat_df['Cluster'] == c]
    plt.scatter(sub['PC1'], sub['PC2'], s=120, color=colors[c], label=cluster_labels[c], alpha=0.7)
    for idx, row in sub.iterrows():
        plt.annotate(idx, (row['PC1'], row['PC2']), fontsize=8, xytext=(5,5), textcoords='offset points')

plt.title('Product Sub-Category Demand Segments (PCA-reduced)', fontsize=13, fontweight='bold')
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('charts/12_clusters.png', dpi=150)
plt.close()

## Task 7 — Model Performance Summary & Recommendations

In [55]:
results_summary = {
    'Model': ['SARIMA', 'Prophet', 'XGBoost'],
    'MAE': [round(sarima_mae, 2), round(prophet_mae, 2), round(xgb_mae, 2)],
    'RMSE': [round(sarima_rmse, 2), round(prophet_rmse, 2), round(xgb_rmse, 2)],
    'MAPE (%)': [round(sarima_mape, 2), round(prophet_mape, 2), round(xgb_mape, 2)]
}
results_df = pd.DataFrame(results_summary)

In [56]:
segment_summary = {
    'Segment': list(segment_results.keys()),
    'Month1_Forecast': [round(v['forecast'][0], 0) for v in segment_results.values()],
    'Month3_Forecast': [round(v['forecast'][2], 0) for v in segment_results.values()],
    'Growth_Pct': [round(v['growth_pct'], 1) for v in segment_results.values()]
}
segment_df = pd.DataFrame(segment_summary)

In [57]:
anomaly_summary = pd.DataFrame({
    'Detection_Method': ['Isolation Forest', 'Z-Score', 'Both Methods'],
    'Count': [weekly_an['iso_flag'].sum(), weekly_an['z_flag'].sum(), (weekly_an['iso_flag'] & weekly_an['z_flag']).sum()]
})

## Task 8 — Supplementary Analysis: Multi-Source Data Integration

In [58]:
vg = pd.read_csv('vgsales.csv')
vg_by_year = vg.groupby('Year')['Global_Sales'].sum().dropna()
vg_by_year = vg_by_year[(vg_by_year.index >= 2000) & (vg_by_year.index <= 2016)]

plt.figure(figsize=(10,5))
plt.plot(vg_by_year.index, vg_by_year.values, marker='o', color='#7c3aed', linewidth=2, markersize=8)
plt.title('Video Game Global Sales by Release Year', fontsize=13, fontweight='bold')
plt.xlabel('Year')
plt.ylabel('Global Sales (millions of units)')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('charts/13_vgsales_supplementary.png', dpi=150)
plt.close()